In [2]:
import re
import numpy as np
from collections import Counter, defaultdict
from datetime import datetime

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
from google.colab import files

uploaded = files.upload()

Saving hostel_bois.txt to hostel_bois.txt


In [4]:
filename = "hostel_bois.txt"

with open(filename, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total Lines :", len(lines))

Total Lines : 3178


In [5]:
messages = []

system_messages = 0

pattern = r"^(\d{2}/\d{2}/\d{2}), (\d{2}):(\d{2}) - (.*)$"

current_message = None

for line in lines:

    line = line.strip()

    if line == "":
        continue

    match = re.match(pattern, line)

    if match:

        date, hour, minute, rest = match.groups()

        if ": " not in rest:
            system_messages += 1
            current_message = None
            continue

        sender, message = rest.split(": ", 1)

        current_message = {
            "date": date,
            "hour": int(hour),
            "minute": int(minute),
            "sender": sender,
            "message": message
        }

        messages.append(current_message)

    else:

        if current_message:
            current_message["message"] += " " + line

In [6]:
print("Total Parsed Messages :", len(messages))

print()

print(messages[0])

Total Parsed Messages : 3174

{'date': '01/04/24', 'hour': 1, 'minute': 17, 'sender': 'Rahul', 'message': 'scene fix'}


In [7]:
user_messages = Counter()

for msg in messages:
    user_messages[msg["sender"]] += 1

print("Messages By User\n")

for user,count in user_messages.items():
    print(user,"-",count)

Messages By User

Rahul - 953
Priya - 718
Karan - 354
Neha - 635
Aman - 490
Vikas - 24


In [8]:
print("="*40)
print("TOP ACTIVE MEMBERS")
print("="*40)

rank = 1

for user,count in user_messages.most_common():

    print(rank,".",user,"-",count)

    rank += 1

TOP ACTIVE MEMBERS
1 . Rahul - 953
2 . Priya - 718
3 . Neha - 635
4 . Aman - 490
5 . Karan - 354
6 . Vikas - 24


In [9]:
print("="*60)
print("TOP 20 MOST FREQUENT WORDS")
print("="*60)

word_counter = Counter()

stop_words = {
    "the","is","am","are","was","were",
    "i","you","he","she","it","they",
    "to","of","and","a","an","in","on",
    "for","my","your","our","we","me",
    "hai","haan","yaar","bhai","okay",
    "this","that","with","have","had"
}

for msg in messages:

    text = msg["message"].lower()

    if text == "<media omitted>":
        continue

    if text == "this message was deleted":
        continue

    words = re.findall(r"[a-zA-Z']+", text)

    for word in words:

        if word not in stop_words and len(word) > 2:
            word_counter[word] += 1

for word,count in word_counter.most_common(20):
    print(f"{word:<20}{count}")

TOP 20 MOST FREQUENT WORDS
how                 321
guys                318
about               274
today               257
his                 217
just                208
which               202
everyone            187
telling             179
from                174
one                 157
started             150
scene               145
entire              145
please              141
anyone              139
but                 138
kya                 133
now                 121
everything          121


In [10]:
print("="*60)
print("MEDIA SHARED")
print("="*60)

media_counter = Counter()

for msg in messages:

    if "<Media omitted>" in msg["message"]:
        media_counter[msg["sender"]] += 1

for user,count in media_counter.items():
    print(user,"-",count)

MEDIA SHARED
Aman - 4
Karan - 7
Neha - 8
Priya - 4
Rahul - 7
Vikas - 2


In [11]:
print("="*60)
print("DELETED MESSAGES")
print("="*60)

deleted_counter = Counter()

for msg in messages:

    if "This message was deleted" in msg["message"]:
        deleted_counter[msg["sender"]] += 1

for user,count in deleted_counter.items():
    print(user,"-",count)

DELETED MESSAGES
Rahul - 6
Karan - 2
Neha - 3
Aman - 2
Priya - 2


In [12]:
print("="*60)
print("HOURLY ACTIVITY")
print("="*60)

hour_counter = Counter()

for msg in messages:

    hour_counter[msg["hour"]] += 1

for hour in range(24):

    print(f"{hour:02d}:00 - {hour_counter[hour]} messages")

HOURLY ACTIVITY
00:00 - 57 messages
01:00 - 82 messages
02:00 - 83 messages
03:00 - 77 messages
04:00 - 110 messages
05:00 - 29 messages
06:00 - 33 messages
07:00 - 55 messages
08:00 - 122 messages
09:00 - 151 messages
10:00 - 160 messages
11:00 - 114 messages
12:00 - 193 messages
13:00 - 159 messages
14:00 - 162 messages
15:00 - 131 messages
16:00 - 189 messages
17:00 - 173 messages
18:00 - 248 messages
19:00 - 228 messages
20:00 - 166 messages
21:00 - 177 messages
22:00 - 116 messages
23:00 - 159 messages


In [14]:
most_hour = hour_counter.most_common(1)

print()

print("Most Active Hour")

print(most_hour)


Most Active Hour
[(18, 248)]


In [15]:
print("="*60)
print("DAY WISE ACTIVITY")
print("="*60)

day_counter = Counter()

for msg in messages:

    date = datetime.strptime(msg["date"], "%d/%m/%y")

    day = date.strftime("%A")

    day_counter[day] += 1

for day,count in day_counter.items():

    print(f"{day:<12}{count}")

DAY WISE ACTIVITY
Monday      482
Tuesday     426
Wednesday   483
Thursday    475
Friday      403
Saturday    459
Sunday      446


In [16]:
print("="*60)
print("AVERAGE MESSAGE LENGTH")
print("="*60)

length = defaultdict(list)

for msg in messages:

    if msg["message"] != "<Media omitted>":

        length[msg["sender"]].append(len(msg["message"]))

for user in length:

    avg = sum(length[user]) / len(length[user])

    print(f"{user:<10}{avg:.2f} characters")

AVERAGE MESSAGE LENGTH
Rahul     10.77 characters
Priya     28.19 characters
Karan     309.76 characters
Neha      25.36 characters
Aman      26.71 characters
Vikas     7.23 characters


In [17]:
print("="*60)
print("24 HOUR HEATMAP")
print("="*60)

heatmap = np.zeros(24)

for msg in messages:

    heatmap[msg["hour"]] += 1

print()

for hour,value in enumerate(heatmap):

    bars = "█" * int(value/10)

    print(f"{hour:02d}:00 {bars}")

24 HOUR HEATMAP

00:00 █████
01:00 ████████
02:00 ████████
03:00 ███████
04:00 ███████████
05:00 ██
06:00 ███
07:00 █████
08:00 ████████████
09:00 ███████████████
10:00 ████████████████
11:00 ███████████
12:00 ███████████████████
13:00 ███████████████
14:00 ████████████████
15:00 █████████████
16:00 ██████████████████
17:00 █████████████████
18:00 ████████████████████████
19:00 ██████████████████████
20:00 ████████████████
21:00 █████████████████
22:00 ███████████
23:00 ███████████████


In [18]:
print("="*60)
print("PERSONALITY REPORT")
print("="*60)

highest = user_messages.most_common(1)[0][1]

lowest = user_messages.most_common()[-1][1]

for user,count in user_messages.items():

    avg = sum(length[user])/len(length[user])

    night = 0

    for msg in messages:

        if msg["sender"] == user:

            if msg["hour"] >= 23 or msg["hour"] <=4:

                night +=1

    if count == highest:

        role = "🔥 Spammer"

    elif count == lowest:

        role = "👻 Silent Observer"

    elif avg > 180:

        role = "📖 Story Teller"

    elif night > 100:

        role = "🌙 Night Owl"

    else:

        role = "😊 Active Member"

    print(f"{user:<10} {role}")

PERSONALITY REPORT
Rahul      🔥 Spammer
Priya      😊 Active Member
Karan      📖 Story Teller
Neha       😊 Active Member
Aman       🌙 Night Owl
Vikas      👻 Silent Observer


In [19]:
print("\n")
print("="*70)
print("          GROUP DNA FINAL REPORT")
print("="*70)

print()

print("Participants          :",len(user_messages))

print("Total Messages        :",len(messages))

print("System Messages       :",system_messages)

print("Media Shared          :",sum(media_counter.values()))

print("Deleted Messages      :",sum(deleted_counter.values()))

print()

print("Most Active Member    :",user_messages.most_common(1)[0][0])

print("Least Active Member   :",user_messages.most_common()[-1][0])

print("Most Active Hour      :",most_hour[0][0],":00")

print()

print("="*70)

print("Project Completed Successfully")

print("="*70)



          GROUP DNA FINAL REPORT

Participants          : 6
Total Messages        : 3174
System Messages       : 4
Media Shared          : 32
Deleted Messages      : 15

Most Active Member    : Rahul
Least Active Member   : Vikas
Most Active Hour      : 18 :00

Project Completed Successfully
